# data exploration and preparation


**Purpose.** The project predicts, the day before, the chance that a scheduled bus departure at a stop
leaves more than 5 minutes late (BO-1) or more than 60 s early (BO-2). The MVP trains and evaluates a
model on one small slice of the data, following CRISP-DM. This notebook covers the *data understanding*
and *data preparation* phases: it loads the slice, explores it, cleans and transforms it, checks its
completeness, and saves one modelling table. Modelling comes in a later notebook.

**Slice.** Route 91 toward St. Norbert, 16 stops, weekdays, scheduled before 18:00,
2025-07 to 2026-11-23. Exported by `probe/export_mvp_slice.py` (`uv run probe/export_mvp_slice.py`).

**Data source.** City of Winnipeg Open Data, *Transit On-Time Performance*:
- archive: <https://data.winnipeg.ca/d/cymk-nyei>
- recent table: <https://data.winnipeg.ca/d/gp3k-am4u>

No GTFS and no weather data are used.

| Field | Type | Meaning |
|---|---|---|
| `route_number` | string | Route, here always `91` |
| `route_destination` | string | Headsign, here always `St. Norbert` |
| `stop_number` | string | Stop ID (16 stops in this slice) |
| `day_type` | string | `Weekday`, `Saturday` or `Sunday`; here always `Weekday` |
| `scheduled_time` | timestamp | Scheduled departure time at the stop (local time) |
| `deviation` | int, seconds | Scheduled minus actual: **negative = late, positive = early** |
| `location` | WKT `POINT (lon lat)` | Stop coordinates |

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Run from the repository root so the data paths below resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")

DATA_PATH = "data/mvp/route91_st_norbert.csv"
PREPARED_PATH = "data/mvp/route91_st_norbert_prepared.parquet"

LATE_S = 300   # BO-1: late by more than 5 min  -> deviation < -300
EARLY_S = 60   # BO-2: early by more than 60 s  -> deviation > 60

pd.set_option("display.max_columns", 50)

## 1. Load data

**Question.** What does the exported slice look like: how many rows, which columns and types?

**Why.** Every later step depends on the raw table loading cleanly with the expected columns and types. Stop numbers are read as strings because they are identifiers, not quantities.

In [ ]:
df = pd.read_csv(DATA_PATH, dtype={"route_number": str, "stop_number": str})
df.shape

In [ ]:
df.dtypes

In [ ]:
df.head()

### Findings (1)

TODO: confirm 80,032 rows × 7 columns, 16 stops, first record 2025-07-04.

## 2. Exploratory data analysis

**Question.** How is `deviation` distributed, and how are the records spread over hours, stops,
months and weekdays? What are the value ranges of each feature?

**Why.** The targets are thresholds on `deviation`, so its shape decides how rare the positive classes
are. The spread over time and stops shows which features carry signal and where data is thin.

### 2.1 Descriptive statistics

In [ ]:
# TODO: df.describe(include="all"); share of rows with deviation < -300 and > 60

### 2.2 Distributions

In [ ]:
# TODO: histogram of deviation (clip the tails for readability), with lines at -300 and +60

In [ ]:
# TODO: number of rows by scheduled hour

In [ ]:
# TODO: number of rows by stop (ordered along the route)

In [ ]:
# TODO: number of rows by month (shows the missing days and the 2026-10-01..09 gap)

In [ ]:
# TODO: number of rows by weekday (Mon..Fri)

### 2.3 Value ranges per feature

In [ ]:
# TODO: min / max / number of distinct values for each column

### Findings (2)

TODO: comment on the shape of `deviation`, class balance of the two targets, and any uneven coverage across hours, stops, months and weekdays.

## 3. Pre-processing

**Question.** Which rows or values must be fixed or removed before features are built?

**Why.** Wrong types, duplicates or impossible deviations would bias both the targets and the lag
features. Each task below states its reason.

### 3.1 Parse timestamps

*Reason:* `scheduled_time` arrives as text; date parts and the service date need a real timestamp.

In [ ]:
# TODO: df["scheduled_time"] = pd.to_datetime(df["scheduled_time"])

### 3.2 Duplicate check

*Reason:* a departure counted twice weighs twice in training. The P0 probe found about 0.03% duplicates on this route.

In [ ]:
# TODO: count exact duplicates and duplicates on (stop_number, scheduled_time); drop them

### 3.3 Drop |deviation| > 3600 s

*Reason:* deviations beyond one hour are recording errors, not service. The rule is kept for reuse on other routes; this slice has **zero** such rows.

In [ ]:
# TODO: mask = df["deviation"].abs() > 3600; report mask.sum() (expected 0); df = df[~mask]

### 3.4 Day-type and weekday checks

*Reason:* the slice should hold only weekdays; a `Weekday` row falling on a Saturday/Sunday, or a statutory holiday served as a weekday, would be mislabelled.

In [ ]:
# TODO: assert day_type == "Weekday"; check scheduled_time.dt.dayofweek < 5; list any holiday dates present

### Findings (3)

TODO: rows removed by each task, and the row count after pre-processing.

## 4. Data transformation

**Question.** Which derived columns does the model need, and how are the two targets defined?

**Why.** The raw table has one timestamp and one deviation. The model needs calendar features,
binary targets, stop position and history, all computable the day before the departure.

### 4.1 Calendar features

*Reason:* lateness varies by time of day, weekday and season.

In [ ]:
# TODO: hour, minute, day_of_week, month, service_date (= scheduled_time.dt.normalize())

### 4.2 Targets

*Reason:* BO-1 and BO-2 are binary questions on `deviation`.

In [ ]:
# TODO: df["late_5min"] = df["deviation"] < -LATE_S
#       df["early_60s"] = df["deviation"] > EARLY_S

### 4.3 Stop coordinates from WKT

*Reason:* position along the route is a numeric feature; WKT text is not.

In [ ]:
# TODO: parse "POINT (lon lat)" into stop_lon, stop_lat

### 4.4 Lagged history features

*Reason:* recent reliability of the same departure and stop is the strongest prior. Rates use data **up to the previous service day only**, so no information from the prediction day leaks in.

In [ ]:
# TODO: per (stop_number, time of day): rolling late / early rates over the last N observed days,
#       shifted by one service day (see section 6 for how missing days are handled)

### Findings (4)

TODO: list the final feature columns and target base rates.

## 5. Standardization

**Question.** Do the numeric features need rescaling, and for which model?

**Why.** A linear comparison model (e.g. logistic regression) is sensitive to feature scale.

> **Note.** Tree models (decision trees, gradient boosting) split on thresholds and are
> scale-invariant, so this step only serves the linear model. The scaled columns are added
> alongside the originals, not in place of them.

### 5.1 Z-score numeric features

*Reason:* put hour, coordinates and lag rates on one scale for the linear model.

In [ ]:
# TODO: z-score numeric features; fit mean/std on the training period only, then apply to all rows

### Findings (5)

TODO: which columns were scaled and the training period used for the fit.

## 6. Data completeness

**Question.** How complete is the slice over time, and how should missing days be treated?

**Why.** The main weakness of this route is whole missing days: data on only **268 of 338** real
weekdays (2025-07-02 to 2026-11-23), while most routes have about 99%. In particular, 2026-10-01 to
2026-10-09 falls between the archive and the recent table. Lag features computed naively over
"the previous row" would silently span these gaps.

### 6.1 Assess completeness

In [ ]:
# TODO: build the list of real weekdays 2025-07-02 .. 2026-11-23 (excluding statutory holidays);
#       count days with data (expected 268 / 338); plot observed vs missing days per month

In [ ]:
# TODO: within observed days, check partial days (fewer departures than the usual daily count)

### 6.2 Treat missing days

*Reason:* the targets of missing days are unknown and must not be imputed; but lag features must know how stale their history is.

In [ ]:
# TODO: build the full calendar x departure grid (weekday dates x (stop, time of day));
#       mark rows from missing days with is_missing_day = True and leave targets NaN

In [ ]:
# TODO: compute lag features from the last *observed* day before each date;
#       add days_since_last_observed as a feature; drop missing-day rows from the modelling table

### Findings (6)

TODO: completeness rate, longest gap, and how the treatment changed the lag features.

## 7. Summary and saved modelling table

**Question.** What table goes into modelling, and what did data understanding and preparation show?

**Why.** The modelling notebook should start from one saved, documented table rather than repeat these steps.

In [ ]:
# TODO: select feature and target columns; df_prepared.to_parquet(PREPARED_PATH, index=False)

### Summary

TODO:
- rows and columns of the prepared table;
- target base rates (`late_5min`, `early_60s`);
- data issues found and how each was treated;
- open questions for modelling.